# Phase 5: generalisation to CAD (Contextual Abuse Dataset, Vidgen et al. 2021)

**Settings:** Accelerator **GPU T4 x2** and **Internet on**. **Attach:** nothing the first time.

The whole method is repeated on a second Reddit dataset: CAD parent→reply pairs, abuse → hate, CounterSpeech → counter,
Neutral/Slur → neutral.
1. Download CAD from Zenodo and build the pairs (CPU, a few minutes). **Check the printed table before the long run.**
2. LLM explanations + stance, LLM zero-shot and 2-shot (2-shot also labels train/val for the hybrid).
3. RoBERTa ablation (lr 1e-5 from Phase 1b): base / +LLM / +label / +LLM+label, 3 seeds.
4. Significance tests + error analysis → `/kaggle/working/outputs/significance_weighted_f1.md`.

**If the session stops** (12 h limit): attach this notebook's last version and run it again. Every step continues where it stopped.

In [ ]:
!nvidia-smi -L
import transformers, torch
print("torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "NO GPU! Settings -> Accelerator -> GPU T4 x2, then run again. (On CPU every run takes hours.)"


In [ ]:
# Clone (or update) the project code and the dataset.
import os, subprocess, glob
REPO = "akashnegetive/stance-aware-llm-explanation-augmented-hate-speech-detection-for-replies-in-conversation"
CODE_DIR = "/kaggle/working/code"
if os.path.isdir(CODE_DIR):
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git", CODE_DIR], check=True)
%cd {CODE_DIR}
if not os.path.isdir("data/counter_context"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/xinchenyu/counter_context", "data/counter_context"], check=True)

In [ ]:
import shlex
CAD = "/kaggle/working/cad_data"            # prepared pairs, kept in this notebook's output
# Earlier versions of this notebook (attach its last version to continue after a stop)
prev = sorted(glob.glob("/kaggle/input/**/outputs", recursive=True))
prev_cad = sorted(glob.glob("/kaggle/input/**/cad_data/gold", recursive=True))
prev_llm = sorted(glob.glob("/kaggle/input/**/llm_cad/llm_outputs.jsonl", recursive=True))
prev_raw = sorted(glob.glob("/kaggle/input/**/llm_cad_*/raw.jsonl", recursive=True))
DIRS = " ".join(shlex.quote(d) for d in prev + ["/kaggle/working/outputs"])
RESUME_RAW = " ".join(shlex.quote(p) for p in prev_raw)
RESUME_LLM = " ".join(shlex.quote(p) for p in prev_llm)
print("continuing from:", prev or "nothing (first run)")

In [ ]:
# 1. CAD -> parent/reply pairs. Reuses the pairs of an earlier run when attached, so the data never changes between sessions.
if prev_cad:
    src_dir = os.path.dirname(prev_cad[0])
    subprocess.run(["cp", "-r", src_dir, CAD], check=True)
    print("reusing pairs from", src_dir)
else:
    # neutral_ratio 1: all hate + counter pairs, and as many neutral pairs (keeps the LLM and training time manageable)
    !python prepare_cad.py --download --cad_dir /tmp/cad_raw --out_dir {CAD} --neutral_ratio 1
!wc -l {CAD}/gold/*.jsonl

In [ ]:
# 2a. LLM explanations + stance for every pair (both GPUs; ~2 pairs/s).
!python generate_llm.py --data_dir {CAD} --out /kaggle/working/llm_cad/llm_outputs.jsonl --resume_from {RESUME_LLM}

In [ ]:
# 2b. LLM-only baselines; 2-shot also labels train/val/test for the hybrid (~1-1.5 pairs/s).
!python llm_classify.py --dataset cad --data_dir {CAD} --shots 0 --output_dir /kaggle/working/outputs --resume_from {RESUME_RAW}
!python llm_classify.py --dataset cad --data_dir {CAD} --shots 2 --splits train val test --output_dir /kaggle/working/outputs --resume_from {RESUME_RAW}

In [ ]:
# 3. RoBERTa ablation on CAD, 3 seeds (one GPU).
LABELS = "/kaggle/working/outputs/llm_cad_Qwen2.5-7B-Instruct_2shot/raw.jsonl"
!CUDA_VISIBLE_DEVICES=0 python ablation.py --dataset cad --data_dir {CAD} --models roberta-base=1e-5 --variants base llm label llm_label --seeds 13 42 87 --select_metric weighted_f1 --llm_file /kaggle/working/llm_cad/llm_outputs.jsonl --llm_label_file {LABELS} --output_dir /kaggle/working/outputs --prev_results {DIRS}

In [ ]:
# 4. Significance tests + error analysis on CAD (CPU, ~2 min).
!python significance.py --results_dirs {DIRS} --data_dir {CAD} --llm_file /kaggle/working/llm_cad/llm_outputs.jsonl --out_dir /kaggle/working/outputs